Setting the python path to import code from app/src.

The extrapath is configured at .vscode/settings.json to avoid IDE errors.

In [ ]:
import sys
from pathlib import Path


def find_project_root(marker="pyproject.toml") -> Path:
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"{marker} not found starting from {p}")


root_dir = find_project_root() / "app" / "src"
sys.path.append(str(root_dir))

#### PDF files and paths

In [ ]:
from infrastructure.parsing.manifest import read_manifest

manifest_path = Path("../../data/policies/manifest.csv")

manifest_info = read_manifest(path=manifest_path)

print(manifest_path)
print(len(manifest_info))

In [ ]:
TARGET_PDF_ID = "1"

target_name = None
for file_info in manifest_info:
    if file_info["id"] == TARGET_PDF_ID:
        target_name = file_info["filename"]
        break

print(target_name)

In [ ]:
pdf_path = Path("../../data/policies/raw/" + target_name)

print(pdf_path)

#### Extraction

In [ ]:
from infrastructure.parsing.extraction import PyMuPdfTextExtractor

pdf_text_extractor = PyMuPdfTextExtractor()

In [ ]:
extracted_document = pdf_text_extractor.extract(
    pdf_path=pdf_path, document_id=TARGET_PDF_ID
)

print(extracted_document)

#### Doc data exploration

In [ ]:
doc_atributes = extracted_document.__dict__.keys()
doc_atributes = list(doc_atributes)

print(doc_atributes)

In [ ]:
print("Document ID: ", extracted_document.document_id)
print("Filename: ", extracted_document.filename)
print("Number of pages: ", len(extracted_document.pages))
print("Extractor version: ", extracted_document.extractor_version)

In [ ]:
TARGET_PAGE_INDEX = 10

selected_page = extracted_document.pages[TARGET_PAGE_INDEX]

print(selected_page)

In [ ]:
page_atributes = selected_page.__dict__.keys()
page_atributes = list(page_atributes)

print(page_atributes)

In [ ]:
print("Page number: ", selected_page.page_number)
print("Number of spans: ", len(selected_page.spans))
print("Char count: ", selected_page.char_count)

In [ ]:
TARGET_SPAN_INDEX = 10

selected_span = selected_page.spans[TARGET_SPAN_INDEX]

print(selected_span)

In [ ]:
span_atributes = selected_span.__dict__.keys()
span_atributes = list(span_atributes)

print(span_atributes)

In [ ]:
print("Document ID: ", selected_span.document_id)
print("Page number: ", selected_span.page_number)
print("Line ID: ", selected_span.line_id)
print("Order: ", selected_span.order)
print("bbox: ", selected_span.bbox)
print("Font size: ", selected_span.font_size)
print("Font name: ", selected_span.font_name)
print("Text: ", selected_span.text)

#### Caching

Only te reading workflow.

In [ ]:
from infrastructure.parsing.caching import read_cache

cache_path = Path("../../data/cache/extraction/1__92b6b4aecbf6b9ad.parquet")
extracted_document = read_cache(cache_path)

print(extracted_document)

In [ ]:
extracted_pages = extracted_document.pages


larger_page = None
for extracted_page in extracted_pages:
    if 2000 <= extracted_page.char_count < 2500:  # to get a filled page
        larger_page = extracted_page
        break

print(larger_page)

In [ ]:
completed_text = ""
for extracted_span in larger_page.spans:
    completed_text += extracted_span.text + "\n"

print(completed_text)